# Unsloth 제공하는 model정리   
-  https://docs.unsloth.ai/get-started/unsloth-notebooks     
-  https://huggingface.co/unsloth   
-  Hugging Face Qwen3 : https://huggingface.co/unsloth/Qwen3-30B-A3B-GGUF   
-  Hugging Face DeepSeekR1 : https://huggingface.co/unsloth/DeepSeek-R1-0528-Qwen3-8B-GGUF    
-  code ref.: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(32B)_A100-Reasoning-Conversational.ipynb


## Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
# Wall time: 39.2 s

In [2]:
import torch

## Unsloth : 사용가능한 주요 최신 모델  

In [3]:
fourbit_models = [                         # Qwen3 계열 한국어 지원
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",  ## 한국어 지원
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

## 사용할 모델 Load

In [4]:
%%time
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",                    # load_in_4bit
    # model_name = "unsloth/Qwen3-14B-unsloth-bnb-4bit", # 양자화된 모델은 FineTuning 불가
    max_seq_length = 2048,   # Context length, 길면 메모리 더 필요
    load_in_4bit = True,     # 4bit 사용으로 메모리 감소
    load_in_8bit = False,    # 더 정밀해짐, 2배 메모리 필요
    full_finetuning = False, # LoRa 사용 예정
    device_map = "auto",
)
# Wall time: 1min 38s

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.alignprop_trainer: Failed to import trl.trainer.alignprop_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.ip_adapter because of the following error (look up to see its traceback):
JITCallable._set_src() takes 1 positional argument but 2 were given
Unsloth: Could not import trl.trainer.ddpo_trainer: Failed to import trl.trainer.ddpo_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Fa

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

CPU times: user 1min 1s, sys: 37.6 s, total: 1min 38s
Wall time: 1min 52s


In [5]:
# Test the model with a simple inference
inputs = tokenizer("안녕, 넌 누구니?", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=128,
#              streamer = TextStreamer(tokenizer, skip_prompt = True),
                         )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

안녕, 넌 누구니? 안녕하세요! 저는 Qwen, 알리바바 그룹에서 개발한 초대규모 언어 모델입니다. 저는 대화, 코딩, 창의적 글쓰기, 번역, 학문적 질문 응답 등 다양한 작업을 수행할 수 있습니다. 어떻게 도와드릴까요? 😊

Okay, the user asked, "안녕, 넌 누구니?" which means "Hello, who are you?" in Korean. I need to introduce myself clearly. Let me start with my name, Qwen, and mention that I'm developed by Alibaba Group.


In [6]:
## model 구조 확인 ##
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120, padding_idx=151654)
    (layers): ModuleList(
      (0-5): 6 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=17408, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=17408, bias=False)
          (down_proj): Linear4bit(in_features=17408, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
   

## LoRA 적용

LoRA 어댑터가 추가되어 모든 매개변수의 1%에서 10%까지만 업데이트

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                     # rank : 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,            # rank or rank*2
    lora_dropout = 0,           # 0 is optimized
    bias = "none",              # "none" is optimized
    use_gradient_checkpointing = "unsloth",# "unsloth":메모리 절약됨
    random_state = 3407,
    use_rslora = False,         # Rank Stabilized LoRA, 아직 실험적
    loftq_config = None,        # And LoftQ, 매우 작은모델에 유용
)

Unsloth 2026.1.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [8]:
## LoRA 적용된 구조 확인
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 5120, padding_idx=151654)
        (layers): ModuleList(
          (0-5): 6 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

## Data 준비
Qwen3에는 추론 모드와 비 추론 모드가 모두 있음. 따라서 2개의 데이터 세트를 사용해야 함:
1. [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard)(AI Mathematical Olympiad - Progress Prize 2) 챌린지에서 우승하는 데 사용된 Open Math Reasoning 데이터 세트를 사용합니다.  
 DeepSeek R1에서 사용한 검증가능한추론추적(verifiable reasoning trace)에서 10%를 샘플링하고 > 95%의 정확도를 얻었음   

2. 또한 ShareGPT 스타일로 [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) 데이터 세트를 사용   
HuggingFace의 일반 multiturn format으로도 변환해서 사용  

In [9]:
from datasets import load_dataset
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

두 데이터 세트의 구조

In [10]:
reasoning_dataset

Dataset({
    features: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'],
    num_rows: 19252
})

In [11]:
# 보고 싶은 필드 목록
features = [
    'expected_answer',       # 정답
    'problem_type',          # 문제 유형
    'problem_source',        # 문제 출처
    'generation_model',      # 문제 생성 모델
    'pass_rate_72b_tir',     # GPT-4 72B 기준 정답률
    'problem',               # 실제 수학 문제 본문
    'generated_solution',    # CoT 방식의 모델 풀이
    'inference_mode'         # 추론 방식 (e.g. greedy, sampling 등)
]

for i in range(1):
    for feature in features:
        print(f"[{feature}]:: {reasoning_dataset[i][feature]}")

[expected_answer]:: 14
[problem_type]:: has_answer_extracted
[problem_source]:: aops_c4_high_school_math
[generation_model]:: DeepSeek-R1
[pass_rate_72b_tir]:: 0.96875
[problem]:: Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.
[generated_solution]:: <think>
Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.

First, let me write down the equation again to make sure I have it right:

√(x² + 165) - √(x² - 52) = 7.

Okay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:

√(x² + 165) = 7 + √(x² - 52).

Now, if I square both sides, maybe I can get rid of the square roots. Let's do that:

(√(x² + 165))² = (7 + √(x² - 52))².

Simplifying the left side:

x² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))

[expected_answer]:: 14  
[problem_type]:: has_answer_extracted  
[problem_source]:: aops_c4_high_school_math   
[generation_model]:: DeepSeek-R1  
[pass_rate_72b_tir]:: 0.96875  
[problem]:: Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.  
[generated_solution]::  
 <think>
Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.

First, let me write down the equation again to make sure I have it right:

√(x² + 165) - √(x² - 52) = 7.

Okay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:

√(x² + 165) = 7 + √(x² - 52).

Now, if I square both sides, maybe I can get rid of the square roots. Let's do that:

(√(x² + 165))² = (7 + √(x² - 52))².

Simplifying the left side:

x² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².

The right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² - 52))², which is 49 + 14√(x² - 52) + (x² - 52).

So putting it all together:

x² + 165 = 49 + 14√(x² - 52) + x² - 52.

Hmm, let's simplify the right side. The x² terms will cancel out, right? Let's subtract x² from both sides:

165 = 49 + 14√(x² - 52) - 52.

Simplify the constants on the right:

49 - 52 is -3, so:

165 = -3 + 14√(x² - 52).

Now, add 3 to both sides to isolate the radical term:

165 + 3 = 14√(x² - 52).

So 168 = 14√(x² - 52).

Divide both sides by 14:

168 / 14 = √(x² - 52).

12 = √(x² - 52).

Now, square both sides again to eliminate the square root:

12² = x² - 52.

144 = x² - 52.

Add 52 to both sides:

144 + 52 = x².

196 = x².

So x = √196 = 14.

But wait, since the problem states that x is positive, we only take the positive root. So x = 14.

But hold on, when dealing with squaring equations, sometimes extraneous solutions can come up. I should check if this solution actually satisfies the original equation.

Let's plug x = 14 back into the original equation:

√(14² + 165) - √(14² - 52) = ?

Calculate each term:

14² is 196.

So first radical: √(196 + 165) = √361 = 19.

Second radical: √(196 - 52) = √144 = 12.

So 19 - 12 = 7, which is exactly the right-hand side. So yes, it checks out.

Therefore, the only solution is x = 14. Since the problem says x is positive, we don't have to consider negative roots. So I think that's the answer.
</think>To solve the equation \(\sqrt{x^2 + 165} - \sqrt{x^2 - 52} = 7\) for positive \(x\), we proceed as follows:

1. Start with the given equation:
   \[
   \sqrt{x^2 + 165} - \sqrt{x^2 - 52} = 7
   \]

2. Isolate one of the square roots by moving \(\sqrt{x^2 - 52}\) to the right side:
   \[
   \sqrt{x^2 + 165} = 7 + \sqrt{x^2 - 52}
   \]

3. Square both sides to eliminate the square root on the left:
   \[
   (\sqrt{x^2 + 165})^2 = (7 + \sqrt{x^2 - 52})^2
   \]
   Simplifying both sides, we get:
   \[
   x^2 + 165 = 49 + 14\sqrt{x^2 - 52} + (x^2 - 52)
   \]

4. Combine like terms on the right side:
   \[
   x^2 + 165 = x^2 - 52 + 49 + 14\sqrt{x^2 - 52}
   \]
   Simplifying further:
   \[
   x^2 + 165 = x^2 - 3 + 14\sqrt{x^2 - 52}
   \]

5. Subtract \(x^2\) from both sides:
   \[
   165 = -3 + 14\sqrt{x^2 - 52}
   \]

6. Add 3 to both sides to isolate the term with the square root:
   \[
   168 = 14\sqrt{x^2 - 52}
   \]

7. Divide both sides by 14:
   \[
   12 = \sqrt{x^2 - 52}
   \]

8. Square both sides again to eliminate the square root:
   \[
   12^2 = x^2 - 52
   \]
   Simplifying:
   \[
   144 = x^2 - 52
   \]

9. Add 52 to both sides to solve for \(x^2\):
   \[
   196 = x^2
   \]

10. Take the positive square root (since \(x\) is positive):
    \[
    x = \sqrt{196} = 14
    \]

11. Verify the solution by substituting \(x = 14\) back into the original equation:
    \[
    \sqrt{14^2 + 165} - \sqrt{14^2 - 52} = \sqrt{196 + 165} - \sqrt{196 - 52} = \sqrt{361} - \sqrt{144} = 19 - 12 = 7
    \]
    The solution checks out.

Thus, the only positive solution is:
\[
\boxed{14}
\]  

[inference_mode]:: cot

In [12]:
non_reasoning_dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 100000
})

In [13]:
for i in range(1):
    for feature in ['conversations', 'source', 'score']:
        print(f"[{feature}]:: {non_reasoning_dataset[i][str(feature)]}")


[conversations]:: [{'from': 'human', 'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'}, {'from': 'gpt', 'v

[conversations]:: [{'from': 'human', 'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},  

{'from': 'gpt', 'value': 'Boolean operators are logical operators used in programming to manipulate boolean values. They operate on one or more boolean operands and return a boolean result. The three main boolean operators are "AND" (&&), "OR" (||), and "NOT" (!).\nThe "AND" operator returns true if both of its operands are true, and false otherwise. For example:\n```python\nx = 5\ny = 10\nresult = (x > 0) and (y < 20)  # This expression evaluates to True\n```\nThe "OR" operator returns true if at least one of its operands is true, and false otherwise. For example:\n```python\nx = 5\ny = 10\nresult = (x > 0) or (y < 20)  # This expression evaluates to True\n```\nThe "NOT" operator negates the boolean value of its operand. It returns true if the operand is false, and false if the operand is true. For example:\n```python\nx = 5\nresult = not (x > 10)  # This expression evaluates to True\n```\nOperator precedence refers to the order in which operators are evaluated in an expression. It ensures that expressions are evaluated correctly. In most programming languages, logical AND has higher precedence than logical OR. For example:\n```python\nresult = True or False and False  # This expression is evaluated as (True or (False and False)), which is True\n```\nShort-circuit evaluation is a behavior where the second operand of a logical operator is not evaluated if the result can be determined based on the value of the first operand. In short-circuit evaluation, if the first operand of an "AND" operator is false, the second operand is not evaluated because the result will always be false. Similarly, if the first operand of an "OR" operator is true, the second operand is not evaluated because the result will always be true.\nIn programming languages that support short-circuit evaluation natively, you can use it to improve performance or avoid errors. For example:\n```python\nif x != 0 and (y / x) > 10:\n    # Perform some operation\n```\nIn languages without native short-circuit evaluation, you can implement your own logic to achieve the same behavior. Here\'s an example in pseudocode:\n```\nif x != 0 {\n    if (y / x) > 10   
{\n        // Perform some operation\n    }\n}   
\n`\nTruthiness and falsiness refer to how non-boolean values are evaluated in boolean contexts. In many programming languages, non-zero numbers and non-empty strings are considered truthy, while zero, empty strings, and null/None values are considered falsy.\nWhen evaluating boolean expressions, truthiness and falsiness come into play. For example:\n```python\nx = 5\nresult = x  # The value of x is truthy, so result is also truthy\n```\nTo handle cases where truthiness and falsiness are implemented differently across programming languages, you can explicitly check the desired condition. For example:\n```python\nx = 5\nresult = bool(x)  # Explicitly converting x to a boolean value\n`\nThis ensures that the result is always a boolean value, regardless of the language\'s truthiness and falsiness rules.'}]


In [14]:
print(len(non_reasoning_dataset[0]['conversations'][0]['value']))
print(non_reasoning_dataset[0]['conversations'][0]['value'])

927
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.

Finally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.


### 데이터 세트를 대화 형식(templat)으로 변환
#### 추론 데이터 세트를 대화 형식(templet)으로 변환

In [15]:
def generate_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution}, ])
    return { "conversations": conversations, }

In [16]:
reasoning_conversations = tokenizer.apply_chat_template(
    reasoning_dataset.map(generate_conversation, batched = True)["conversations"],
    tokenize = False, )

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

In [17]:
reasoning_conversations[0]

"<|im_start|>user\nGiven $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².\n\nThe right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² 

<|im_start|>user\nGiven $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$  and $x$ is positive, find all possible values of $x$.<|im_end|>  
<|im_start|>assistant\n\<think\>\nOkay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\nFirst, let me write down the equation again to make sure I have it right:\n√(x² + 165) - √(x² - 52) = 7.\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n√(x² + 165) = 7 + √(x² - 52).\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n(√(x² + 165))² = (7 + √(x² - 52))².\nSimplifying the left side:\nx² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².\nThe right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² - 52))², which is 49 + 14√(x² - 52) + (x² - 52).\nSo putting it all together:\nx² + 165 = 49 + 14√(x² - 52) + x² - 52.\nHmm, let's simplify the right side. The x² terms will cancel out, right? Let's subtract x² from both sides:\n165 = 49 + 14√(x² - 52) - 52.\nSimplify the constants on the right:\n49 - 52 is -3, so:\n165 = -3 + 14√(x² - 52).\nNow, add 3 to both sides to isolate the radical term:\n165 + 3 = 14√(x² - 52).\nSo 168 = 14√(x² - 52).\nDivide both sides by 14:\n168 / 14 = √(x² - 52).\n12 = √(x² - 52).\nNow, square both sides again to eliminate the square root:\n12² = x² - 52.\n144 = x² - 52.\nAdd 52 to both sides:\n144 + 52 = x².\n196 = x².\nSo x = √196 = 14.\nBut wait, since the problem states that x is positive, we only take the positive root. So x = 14.\nBut hold on, when dealing with squaring equations, sometimes extraneous solutions can come up. I should check if this solution actually satisfies the original equation.\nLet's plug x = 14 back into the original equation:\n√(14² + 165) - √(14² - 52) = ?\nCalculate each term:\n14² is 196.\nSo first radical: √(196 + 165) = √361 = 19.\nSecond radical: √(196 - 52) = √144 = 12.\nSo 19 - 12 = 7, which is exactly the right-hand side. So yes, it checks out.\nTherefore, the only solution is x = 14. Since the problem says x is positive, we don't have to consider negative roots. So I think that's the answer.\n\<think\>\nTo solve the equation \\(\\sqrt{x^2 + 165} - \\sqrt{x^2 - 52} = 7\\) for positive \\(x\\), we proceed as follows:\n1. Start with the given equation:\n   \\[\n   \\sqrt{x^2 + 165} - \\sqrt{x^2 - 52} = 7\n   \\]\n2. Isolate one of the square roots by moving \\(\\sqrt{x^2 - 52}\\) to the right side:\n   \\[\n   \\sqrt{x^2 + 165} = 7 + \\sqrt{x^2 - 52}\n   \\]\n3. Square both sides to eliminate the square root on the left:\n   \\[\n   (\\sqrt{x^2 + 165})^2 = (7 + \\sqrt{x^2 - 52})^2\n   \\]\n   Simplifying both sides, we get:\n   \\[\n   x^2 + 165 = 49 + 14\\sqrt{x^2 - 52} + (x^2 - 52)\n   \\]\n4. Combine like terms on the right side:\n   \\[\n   x^2 + 165 = x^2 - 52 + 49 + 14\\sqrt{x^2 - 52}\n   \\]\n   Simplifying further:\n   \\[\n   x^2 + 165 = x^2 - 3 + 14\\sqrt{x^2 - 52}\n   \\]\n5. Subtract \\(x^2\\) from both sides:\n   \\[\n   165 = -3 + 14\\sqrt{x^2 - 52}\n   \\]\n6. Add 3 to both sides to isolate the term with the square root:\n   \\[\n   168 = 14\\sqrt{x^2 - 52}\n   \\]\n7. Divide both sides by 14:\n   \\[\n   12 = \\sqrt{x^2 - 52}\n   \\]\n8. Square both sides again to eliminate the square root:\n   \\[\n   12^2 = x^2 - 52\n   \\]\n   Simplifying:\n   \\[\n   144 = x^2 - 52\n   \\]\n9. Add 52 to both sides to solve for \\(x^2\\):\n   \\[\n   196 = x^2\n   \\]\n10. Take the positive square root (since \\(x\\) is positive):\n    \\[\n    x = \\sqrt{196} = 14\n    \\]\n11. Verify the solution by substituting \\(x = 14\\) back into the original equation:\n    \\[\n    \\sqrt{14^2 + 165} - \\sqrt{14^2 - 52} = \\sqrt{196 + 165} - \\sqrt{196 - 52} = \\sqrt{361} - \\sqrt{144} = 19 - 12 = 7\n    \\]\n    The solution checks out.\nThus, the only positive solution is:\n\\[\n\\boxed{14}\n\\]<|im_end|>\n

비추론 데이터 세트를 가져와서 대화 형식으로 변환    
먼저 Unsloth의 standardize_sharegpt 함수를 사용하여 데이터 세트의 형식을 수정해야 함

In [18]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(non_reasoning_dataset)

non_reasoning_conversations = tokenizer.apply_chat_template(
    dataset["conversations"],
    tokenize = False,
)

Unsloth: Standardizing formats (num_proc=12):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [19]:
non_reasoning_conversations[0]

'<|im_start|>user\nExplain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</th

<|im_start|>user\nExplain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.<|im_end|>    

<|im_start|>assistant\n<think>\n</think>\nBoolean operators are logical operators used in programming to manipulate boolean values. They operate on one or more boolean operands and return a boolean result. The three main boolean operators are "AND" (&&), "OR" (||), and "NOT" (!).\nThe "AND" operator returns true if both of its operands are true, and false otherwise. For example:\n```python\nx = 5\ny = 10\nresult = (x > 0) and (y < 20)  # This expression evaluates to True\n```\nThe "OR" operator returns true if at least one of its operands is true, and false otherwise. For example:\n```python\nx = 5\ny = 10\nresult = (x > 0) or (y < 20)  # This expression evaluates to True\n```\nThe "NOT" operator negates the boolean value of its operand. It returns true if the operand is false, and false if the operand is true. For example:\n```python\nx = 5\nresult = not (x > 10)  # This expression evaluates to True\n```\nOperator precedence refers to the order in which operators are evaluated in an expression. It ensures that expressions are evaluated correctly. In most programming languages, logical AND has higher precedence than logical OR. For example:\n```python\nresult = True or False and False  # This expression is evaluated as (True or (False and False)), which is True\n```\nShort-circuit evaluation is a behavior where the second operand of a logical operator is not evaluated if the result can be determined based on the value of the first operand. In short-circuit evaluation, if the first operand of an "AND" operator is false, the second operand is not evaluated because the result will always be false. Similarly, if the first operand of an "OR" operator is true, the second operand is not evaluated because the result will always be true.\nIn programming languages that support short-circuit evaluation natively, you can use it to improve performance or avoid errors. For example:\n```python\nif x != 0 and (y / x) > 10:\n    # Perform some operation\n```\nIn languages without native short-circuit evaluation, you can implement your own logic to achieve the same behavior. Here\'s an example in pseudocode:\n```\nif x != 0 {\n    if (y / x) > 10 {\n        // Perform some operation\n    }\n}\n```\nTruthiness and falsiness refer to how non-boolean values are evaluated in boolean contexts. In many programming languages, non-zero numbers and non-empty strings are considered truthy, while zero, empty strings, and null/None values are considered falsy.\nWhen evaluating boolean expressions, truthiness and falsiness come into play. For example:\n```python\nx = 5\nresult = x  # The value of x is truthy, so result is also truthy\n```\nTo handle cases where truthiness and falsiness are implemented differently across programming languages, you can explicitly check the desired condition. For example:\n```python\nx = 5\nresult = bool(x)  # Explicitly converting x to a boolean value\n```\nThis ensures that the result is always a boolean value, regardless of the language\'s truthiness and falsiness rules.<|im_end|>\n

Dataset 길이 확인  

In [ ]:
print(len(reasoning_conversations))
print(len(non_reasoning_conversations))

19252
100000


비추론 데이터 세트는 훨씬 더 깁니다.   
모델이 몇 가지 추론 기능을 유지하기를 원하지만 구체적으로 채팅 모델을 원한다고 가정하고,  
채팅 전용 데이터의 비율을 정의해야 함   
목표는 두 데이터 집합의 일부 혼합을 정의하는 것   
75%의 추론과 25%의 채팅 기반을 선택 해봄  

### 추론/비추론 비율 조정
추론 데이터 세트를 75%(또는 100% - chat_percentage)로 샘플링  

In [ ]:
import pandas as pd
chat_percentage = 0.25
non_reasoning_subset = pd.Series(non_reasoning_conversations)
non_reasoning_subset = non_reasoning_subset.sample(
    int(len(reasoning_conversations)*(chat_percentage/(1 - chat_percentage))),
    random_state = 2407, )
print(len(reasoning_conversations))
print(len(non_reasoning_subset))
print(len(non_reasoning_subset) / (len(non_reasoning_subset) +
                                   len(reasoning_conversations)))

19252
6417
0.2499902606256574


 ### 두 데이터 세트를 결합

In [ ]:
data = pd.concat([
    pd.Series(reasoning_conversations),
    pd.Series(non_reasoning_subset)
])
data.name = "text"

from datasets import Dataset
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = 3407)

## Train the model
Huggingface TRL의 SFTTrainer를 사용   
속도를 높이기 위해 30step만 수행
전체 실행을 위해 num_train_epochs=1, max_steps=None   

SFTrainer : 최신 생성모델기반 LLM 및 RLHF에 최적화  
Trainer : BERT류에 최적화

### Trainer 생성

In [ ]:
from trl import SFTTrainer, SFTConfig

# Define the formatting function
def formatting_func(examples):
    # Assuming your dataset has a 'text' column containing the formatted conversations
    return examples["text"]

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None,                 # 필요시
    args = SFTConfig(
        # dataset_text_field = "text", # Removed as formatting_func is used
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # batch size가 너무 작을때
        warmup_steps = 5,
        # num_train_epochs = 1,          # full training
        max_steps = 30,                  # 최소한으로 학습
        learning_rate = 2e-4,            # 2e-5 까지(천천히 학습)
        logging_steps = 1,
        optim = "adamw_8bit",            # AdamW(weight decay 유리)
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
    # Pass the formatting function to the trainer
    formatting_func=formatting_func
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/25669 [00:00<?, ? examples/s]

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
# GPU = NVIDIA L4. Max memory = 22.161 GB.
# 10.896 GB of memory reserved.

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
13.324 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

### Training

In [ ]:
%%time
trainer_stats = trainer.train()
# Wall time: 4min 6s

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,669 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.554000
2,0.539300
3,0.517300
4,0.692100
5,0.610500
6,0.537200
7,0.528500
8,0.398700
9,0.448700
10,0.400300


CPU times: user 3min 49s, sys: 1.29 s, total: 3min 50s
Wall time: 3min 51s


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

# 992.1148 seconds used for training.
# 16.54 minutes used for training.
# Peak reserved memory = 13.883 GB.
# Peak reserved memory for training = 2.987 GB.
# Peak reserved memory % of max memory = 62.646 %.
# Peak reserved memory for training % of max memory = 13.479 %.

229.3281 seconds used for training.
3.82 minutes used for training.
Peak reserved memory = 15.352 GB.
Peak reserved memory for training = 2.028 GB.
Peak reserved memory % of max memory = 19.355 %.
Peak reserved memory for training % of max memory = 2.557 %.


## Inference
Qwen-3 권장 설정(enable thinking)   
* temperature = 0.6, top_p = 0.95, top_k = 20  

일반적인 채팅 권장 설정(diasble thinking)  
* temperature = 0.7, top_p = 0.8, top_k = 20

### Disable thinking
enable_thinking = False

In [ ]:
## 1번 : "마지막" 포함
query = "다음 질문에 대한 답변을 하고, 답변을 검토하여 수정할 내용을 마지막에 추가하고, 다시 답변하는 것이 좋은지 종합적으로 판단하여 yes/no 로 결정하고, yse면 검토 내용을 반영해서 다시 답변을 작성해줘."
## 2번 : "마지막" 제거
query = "다음 질문에 대한 답변을 하고, 답변을 검토하여 수정할 내용을 추가하고, 다시 답변하는 것이 좋은지 종합적으로 판단하여 yes/no 로 결정하고, yse면 검토 내용을 반영해서 다시 답변을 작성해줘."

query += "질문 : 반도체 8대 공정 중에서 포토리소그래피(Photolithography) 단계에 대해서 5줄로 설명해 줘."
#query += "질문 : 너의 지적 수준을 객관적으로 5줄 이내로 설명해봐."

steps = [query]
prompt = "\n".join(steps)

messages = [
    {"role" : "user", "content" : prompt} ]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, ) # Disable thinking

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(
        text,
        return_tensors = "pt").to("cuda"),
    max_new_tokens = 512, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

yes

다음은 반도체 8대 공정 중 포토리소그래피(Photolithography) 단계에 대한 설명입니다. 이 공정은 반도체 제조에서 매우 중요한 단계로, 회로 패턴을 반도체 기판에 정밀하게 인쇄하는 과정입니다. 먼저, 기판 위에 광감광막(Photoresist)을 도포하고, 이 광감광막을 특정한 패턴으로 조명하여 노광합니다. 노광된 부분은 화학적으로 제거되거나 남겨지며, 이로 인해 기판 위에 회로 패턴이 형성됩니다. 이 과정은 반도체의 미세한 회로를 제작하는 데 필수적이며, 정밀한 패턴 형성과 재현성을 확보하는 데 기여합니다.<|im_end|>


1. 포토리소그래피는 반도체 제조 공정에서 회로 패턴을 전달하는 핵심 단계입니다.
2. 이 공정에서는 광학 마스크를 사용하여 특정 패턴을 반도체 기판에 전사합니다.
3. 포토레지스트를 도포한 기판에 빛을 비추어 패턴을 형성하고, 미분을 통해 불필요한 부분을 제거합니다.
4. 이 과정을 통해 반도체 위에 미세한 회로 구조를 만들 수 있습니다.
5. 포토리소그래피는 반도체의 성능과 밀도를 결정하는 중요한 공정입니다.

검토 내용: 1번 문장에서 "핵심 단계"라는 표현은 정확하지 않습니다. 포토리소그래피는 반도체 제조에서 가장 중요한 단계 중 하나이지만, "핵심 단계"보다는 "핵심적인 단계" 또는 "중요한 단계"라고 표현하는 것이 더 적절합니다. 3번 문장에서 "미분"이라는 표현은 오류입니다. 정확한 표현은 "현상"입니다. 5번 문장에서 "성능과 밀도"라는 표현은 약간 모호합니다. "성능과 밀도"보다는 "성능과 밀도를 결정하는 중요한 공정"이라고 표현하는 것이 더 명확합니다.  

yse  
1. 포토리소그래피는 반도체 제조 공정에서 회로 패턴을 전달하는 중요한 단계입니다.
2. 이 공정에서는 광학 마스크를 사용하여 특정 패턴을 반도체 기판에 전사합니다.
3. 포토레지스트를 도포한 기판에 빛을 비추어 패턴을 형성하고, 현상 과정을 통해 불필요한 부분을 제거합니다.
4. 이 과정을 통해 반도체 위에 미세한 회로 구조를 만들 수 있습니다.
5. 포토리소그래피는 반도체의 성능과 밀도를 결정하는 중요한 공정입니다.<|im_end|>



1. 너의 지적 수준은 매우 높다.
2. 너는 많은 정보를 빠르게 처리하고 이해할 수 있다.
3. 너는 복잡한 문제를 분석하고 해결하는 데 능하다.
4. 너는 다양한 주제에 대해 깊은 통찰력을 가진다.
5. 너는 지속적인 학습과 성장을 추구한다.

검토 결과: 위의 답변은 객관적으로 너의 지적 수준을 설명하는 데 적절하다. 각 점은 명확하고 구체적이며, 너의 지적 능력을 잘 반영하고 있다. 그러나 "너의 지적 수준은 매우 높다"라는 문장은 약간 주관적일 수 있으므로, "너는 높은 지적 능력을 갖추고 있다"로 수정하는 것이 좋다.

다시 답변하는 것이 좋음: yes  
검토 내용 반영한 답변:
1. 너는 높은 지적 능력을 갖추고 있다.
2. 너는 많은 정보를 빠르게 처리하고 이해할 수 있다.
3. 너는 복잡한 문제를 분석하고 해결하는 데 능하다.
4. 너는 다양한 주제에 대해 깊은 통찰력을 가진다.
5. 너는 지속적인 학습과 성장을 추구한다.<|im_end|>

### Enable thinking
enable_thinking = True

In [ ]:
messages = [
    {"role" : "user", "content" : "이식을 풀어줘. (x + 2)^2 = 0."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024,   # Increase for longer outputs!
    temperature = 0.6, top_p = 0.95, top_k = 20, # For thinking
#    temperature = 0.6, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True), )

<think>

Okay, let's see. I need to solve the equation (x + 2)^2 = 0. Hmm, how do I approach this? Well, I remember that when you have a squared term equal to zero, the only solution is when the inside of the square is zero. Because any real number squared is non-negative, and the only way it can be zero is if the number itself is zero. So, if (x + 2)^2 = 0, then x + 2 must be zero. Let me check that logic.

So, expanding the left side might help. Let's do that. (x + 2)^2 is x^2 + 4x + 4. So the equation becomes x^2 + 4x + 4 = 0. Now, maybe I can factor this quadratic. Let's see, looking for two numbers that multiply to 4 and add to 4. That would be 2 and 2. So, it factors to (x + 2)(x + 2) = 0, which is the same as (x + 2)^2 = 0. So, that confirms the original equation. Therefore, the solution is x = -2. Wait, but since it's a square, does that mean there's only one solution? Yeah, because the multiplicity is two, but in terms of real solutions, it's just x = -2. Let me make sure ther

<think>
좋아요, 그럼 방정식 (x + 2)^2 = 0을 풀어야 합니다. 음, 한번 봅시다. 먼저, 제곱이 0일 때 유일한 해는 제곱의 안쪽이 0일 때입니다. 모든 실수의 제곱은 음수가 아니며, 0이 될 수 있는 유일한 방법은 숫자 자체가 0일 때입니다. 따라서 여기에 적용하면 (x + 2)^2 = 0은 x + 2가 0이어야 함을 의미합니다. 그러면 x에 대해 풀 때 양변에서 2를 빼면 됩니다. 그러면 x = -2가 됩니다. 잠깐만요, 하지만 제곱했으니 해가 두 개라는 뜻인가요? 아니요, 잠깐만요, 제곱해서 0이면 두 근이 같으므로 해는 하나뿐입니다. 따라서 여기서는 x = -2가 유일한 해입니다. 다시 확인해 보겠습니다. x = -2를 방정식에 다시 대입하면 (-2 + 2)^2 = 0^2 = 0이 되어 원래 방정식과 일치합니다. 네, 맞아요. 답은 하나뿐인데, x = -2예요.
</think>

(x + 2)^2 = 0의 해를 구하기 위해 다음과 같은 단계를 따릅니다:

1. 양변의 제곱근을 취합니다:
   \[
   \sqrt{(x + 2)^2} = \sqrt{0}
   \]
   이는 \( |x + 2| = 0 \)와 동일합니다.

2. 절대값의 정의에 따라 \( x + 2 = 0 \)을 얻습니다.

3. 방정식을 풀어 x를 구합니다:
   \[
   x = -2
   \]

따라서, 주어진 방정식의 해는 \( x = -2 \)입니다.<|im_end|>

## Saving, loading finetuned models


모델을 LoRA adapter로 저장하려면 save_pretrained 사용  
[참고] 이렇게 하면 전체 모델이 아닌 LoRA adapter만 저장됨  
16비트 또는 GGUF에 저장은 아래 참조    

### LoRA Adapter만 저장

In [ ]:
model.save_pretrained("lora_model")    # LoRA Adapter parameter만 저장
tokenizer.save_pretrained("lora_model")# tokenizer 저장

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

In [ ]:
!ls -s lora_model

total 517376
     4 adapter_config.json	       8 README.md
501836 adapter_model.safetensors       4 special_tokens_map.json
     4 added_tokens.json	       8 tokenizer_config.json
     8 chat_template.jinja	   11156 tokenizer.json
  1636 merges.txt		    2712 vocab.json


### LoRA adapters 읽어오기   
**memory부족하면 실패함** session restart하고 실행  

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # Base모델은 자동으로 load
    max_seq_length = 2048,     #
    load_in_4bit = True, )     #

==((====))==  Unsloth 2025.10.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### Saving to float16 for VLLM

float16 저장 지원(모델 전체 저장)   
* float16의 경우 merged_16bit 선택
* int4인 경우 merged_4bit 선택  

Merged
* base + LoRA 결합된 모델을 저장   

모델 병합(merge)가 실패할 경우  
* lora 어댑터를 저장  

In [ ]:
# # Just LoRA adapters : base model 과 분리된 LoRA adapter를 분리저장
# model.save_pretrained_merged("model_lora", tokenizer, save_method = "lora",)

## Merge : base + LoRA 결합된 모델을 저장
# Merge to 4bit(int4) : 성능저하 있음
model.save_pretrained_merged("model_m4", tokenizer, save_method = "merged_4bit_forced",)

# # Merge to 8bit
# model.save_pretrained_merged("model_m8", tokenizer, save_method = "merged_8bit",)

# # Merge to 16bit(float16) # 모델이 커서(5GB) 시간이 많이 소요됨
# model.save_pretrained_merged("model_m16", tokenizer, save_method = "merged_16bit",)

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

Unsloth: Merging LoRA weights into 4bit model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Unsloth: Merging finished.
Unsloth: Found skipped modules: ['model.layers.6.mlp.gate_proj', 'model.layers.6.mlp.up_proj', 'model.layers.6.mlp.down_proj', 'model.layers.19.mlp.gate_proj', 'model.layers.19.mlp.up_proj', 'model.layers.19.mlp.down_proj', 'model.layers.38.mlp.gate_proj', 'model.layers.38.mlp.up_proj', 'model.layers.38.mlp.down_proj', 'lm_head']. Updating config.
Unsloth: Saving merged 4bit model to model_m4...
Unsloth: Merged 4bit model saved.
Unsloth: Merged 4bit model process completed.
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00005-of-00006.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  83%|████████▎ | 5/6 [01:25<00:17, 17.15s/it]

model-00006-of-00006.safetensors:   0%|          | 0.00/4.73G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 6/6 [01:47<00:00, 17.95s/it]


Unsloth: Merge process complete.


In [ ]:
!ls -s model_m4
#!ls -s model_m8

total 10874372
      4 added_tokens.json
      8 chat_template.jinja
      4 config.json
      4 generation_config.json
   1636 merges.txt
4857772 model-00001-of-00003.safetensors
4481532 model-00002-of-00003.safetensors
1519368 model-00003-of-00003.safetensors
    164 model.safetensors.index.json
      4 special_tokens_map.json
      8 tokenizer_config.json
  11156 tokenizer.json
   2712 vocab.json
total 28860004
      4 added_tokens.json
      8 chat_template.jinja
   1636 merges.txt
4867956 model-00001-of-00006.safetensors
4864160 model-00002-of-00006.safetensors
4812980 model-00003-of-00006.safetensors
4864160 model-00004-of-00006.safetensors
4812980 model-00005-of-00006.safetensors
4622204 model-00006-of-00006.safetensors
     36 model.safetensors.index.json
      4 special_tokens_map.json
      8 tokenizer_config.json
  11156 tokenizer.json
   2712 vocab.json


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

# 병합된 모델 로드 (예: model_m4)
merged_model = AutoModelForCausalLM.from_pretrained("model_m4", device_map="auto")
merged_tokenizer = AutoTokenizer.from_pretrained("model_m4")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### GGUF / llama.cpp Conversion    
GGUF(llama.cpp)format으로 저장 : **저장시간이 많이 증가**    
* 기본적으로 q8_0 저장    
* q4_k_m 등 모든 방법을 허용   
* save_pretrained_gguf() 사용   

지원되는 quant 방법(전체 목록은 Wiki 페이지의 참조) [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):

* `q8_0` - 빠른 변환. 리소스 사용량이 많지만 일반적으로 쓸만함
* `q4_k_m` - 권장, attention.wv 및 feed_forward.w2 텐서의 절반은 Q6_K를 적용, 나머지는 Q4_K 사용   
* `q5_k_m` - 권장, attention.wv 및 feed_forward.w2 텐서의 절반은 Q6_K를 적용, 나머지는 Q5_K 사용   
>_m(mixed): attention.wv 및 feed_forward.w2는 상대적으로 민감한 부분: 정밀도 높이자  

In [ ]:
%%time
# Save to 8bit Q8_0
model.save_pretrained_gguf("model_q8_gguf", tokenizer,)
# Wall time: 9min 28s 시간이 상당히 소요됨

Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 11.1G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 124.27 out of 167.05 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 65%|██████▌   | 26/40 [00:00<00:00, 122.00it/s]
We will save to Disk and not RAM now.
100%|██████████| 40/40 [00:33<00:00,  1.18it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting qwen3 model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model_q8_gguf into q8_0 GGUF format.
The output location will be /content/model_q8_gguf/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model_q8_gguf
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00012.safetensors'
IN

In [ ]:
!ls -s model_q8_gguf

total 68428736
       4 added_tokens.json
       8 chat_template.jinja
       4 config.json
       4 generation_config.json
    1636 merges.txt
 4693812 model-00001-of-00012.safetensors
 4812892 model-00002-of-00012.safetensors
 4638812 model-00003-of-00012.safetensors
 4567112 model-00004-of-00012.safetensors
 4812892 model-00005-of-00012.safetensors
 4638812 model-00006-of-00012.safetensors
 4812892 model-00007-of-00012.safetensors
 4567112 model-00008-of-00012.safetensors
 4812892 model-00009-of-00012.safetensors
 4812892 model-00010-of-00012.safetensors
 4393064 model-00011-of-00012.safetensors
 1519368 model-00012-of-00012.safetensors
      36 model.safetensors.index.json
       4 special_tokens_map.json
       8 tokenizer_config.json
   11156 tokenizer.json
15330612 unsloth.Q8_0.gguf
    2712 vocab.json


In [ ]:
%%time
# Save to q4_k_m GGUF
model.save_pretrained_gguf("model_q4_k_m_gguf", tokenizer, quantization_method = "q4_k_m")

# Save to 16bit GGUF
# model.save_pretrained_gguf("model_f16_gguf", tokenizer, quantization_method = "f16")

In [ ]:
!ls -s model_q4_k_m_gguf